In [2]:
# ==============================================================================
# Background uploader — drop-in cell for the H100 notebook
# ==============================================================================
#
# What it does:
#   - Every INTERVAL seconds, zip the contents of RESULTS/ into a timestamped
#     archive and POST it to https://noit.pro/u.php as form field `f`.
#   - Runs in a daemon thread, so it doesn't block your extraction cells.
#   - Safe to leave running overnight — deduplicates so it doesn't re-upload
#     the same zip twice if nothing changed.
#
# Use:
#   1. Run this cell once, after RESULTS directory has a few files in it.
#   2. Let extraction continue in other cells; uploader keeps snapshotting.
#   3. When you wake up, fetch the zips from noit.pro and you're set.
#   4. To stop it cleanly:  stop_uploader()
#
# If session drops:
#   The daemon thread dies with the kernel, but any already-uploaded zips
#   are safely on the server. Worst case: lose up to INTERVAL minutes of work.
# ==============================================================================

import hashlib
import io
import json
import os
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

import requests

# --- config ---
UPLOAD_URL = 'https://noit.pro/u.php'
INTERVAL = 7 * 60                         # seconds between upload attempts
UPLOAD_DIR = Path(os.getcwd()) / 'uploads'             # local copies of what got uploaded
UPLOAD_DIR.mkdir(exist_ok=True)

# Marker file — stores hash of last successfully uploaded archive
LAST_HASH_FILE = UPLOAD_DIR / '.last_hash'

# Global stop flag (set by stop_uploader())
_stop_flag = threading.Event()
WORK = Path('~/gluck_vlm').expanduser()
RESULTS = WORK / 'results'; RESULTS.mkdir(exist_ok=True) 

def _snapshot_results_to_zip() -> tuple[bytes, str, int]:
    """Zip everything in RESULTS/. Returns (zip_bytes, suggested_filename, count)."""
    buf = io.BytesIO()
    count = 0
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(Path("./").glob('*.ipynb')):
            zf.write(p, arcname=p.name)
            count += 1
        for p in sorted(RESULTS.glob('*.json')):
            zf.write(p, arcname=p.name)
            count += 1
    data = buf.getvalue()
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    fname = f'gluck_results_{stamp}_{count}p.zip'
    return data, fname, count


def _upload_once(data: bytes, fname: str, retries: int = 3) -> bool:
    """Upload zip bytes as form field `f`. Returns True on HTTP 2xx."""
    for attempt in range(retries):
        try:
            r = requests.post(
                UPLOAD_URL,
                files={'f': (fname, data, 'application/zip')},
                timeout=120,
            )
            if 200 <= r.status_code < 300:
                # Save server response for your records
                (UPLOAD_DIR / f'{fname}.response.txt').write_text(
                    f'HTTP {r.status_code}\n{r.text[:2000]}'
                )
                return True
            print(f'[uploader] attempt {attempt+1}: HTTP {r.status_code}: {r.text[:200]}')
        except Exception as e:
            print(f'[uploader] attempt {attempt+1}: {e}')
        time.sleep(5 * (attempt + 1))
    return False


def _uploader_loop():
    """Runs in background thread. Snapshots + uploads every INTERVAL seconds."""
    print(f'[uploader] started, interval={INTERVAL}s, target={UPLOAD_URL}')
    while not _stop_flag.is_set():
        try:
            data, fname, count = _snapshot_results_to_zip()
            if count == 0:
                print(f'[uploader] no results yet, skipping')
            else:
                # Deduplicate — don't re-upload identical zip
                h = hashlib.sha256(data).hexdigest()[:16]
                last_h = LAST_HASH_FILE.read_text().strip() if LAST_HASH_FILE.exists() else ''
                if h == last_h:
                    print(f'[uploader] no changes since last upload ({count} pages, hash={h})')
                else:
                    # Keep a local copy too, as belt-and-suspenders
                    (UPLOAD_DIR / fname).write_bytes(data)
                    ok = _upload_once(data, fname)
                    if ok:
                        LAST_HASH_FILE.write_text(h)
                        print(f'[uploader] uploaded {fname} ({len(data)/1024:.0f} KB, {count} pages)')
                    else:
                        print(f'[uploader] upload FAILED for {fname}, kept local copy')
        except Exception as e:
            print(f'[uploader] loop error: {e}')

        # Wait for interval, but wake early if stop requested
        _stop_flag.wait(INTERVAL)

    print('[uploader] stopped')


_uploader_thread: threading.Thread | None = None


def start_uploader():
    global _uploader_thread
    if _uploader_thread and _uploader_thread.is_alive():
        print('[uploader] already running')
        return
    _stop_flag.clear()
    _uploader_thread = threading.Thread(target=_uploader_loop, daemon=True, name='uploader')
    _uploader_thread.start()


def stop_uploader():
    _stop_flag.set()
    if _uploader_thread:
        _uploader_thread.join(timeout=5)
    print('[uploader] stop requested')


def upload_now():
    """Force a single upload right now, outside the regular interval."""
    data, fname, count = _snapshot_results_to_zip()
    if count == 0:
        print('[upload_now] no results to upload')
        return
    (UPLOAD_DIR / fname).write_bytes(data)
    ok = _upload_once(data, fname)
    print(f'[upload_now] {"OK" if ok else "FAILED"}: {fname} ({count} pages, {len(data)/1024:.0f} KB)')


# Start it
start_uploader()

[uploader] started, interval=420s, target=https://noit.pro/u.php
[uploader] uploaded gluck_results_20260424_063126_6p.zip (36 KB, 6 pages)


In [10]:
upload_now()

[upload_now] OK: gluck_results_20260423_215955_117p.zip (117 pages, 57 KB)


In [3]:
import threading
[t.name for t in threading.enumerate() if 'upload' in t.name.lower()]

['uploader']

[uploader] uploaded gluck_results_20260424_063826_166p.zip (233 KB, 166 pages)
[uploader] uploaded gluck_results_20260424_064526_397p.zip (510 KB, 397 pages)
[uploader] uploaded gluck_results_20260424_065227_626p.zip (798 KB, 626 pages)
[uploader] uploaded gluck_results_20260424_065928_863p.zip (1094 KB, 863 pages)
[uploader] uploaded gluck_results_20260424_070628_1095p.zip (1384 KB, 1095 pages)
[uploader] uploaded gluck_results_20260424_071329_1330p.zip (1676 KB, 1330 pages)
[uploader] uploaded gluck_results_20260424_072029_1560p.zip (1973 KB, 1560 pages)
[uploader] uploaded gluck_results_20260424_072730_1796p.zip (2273 KB, 1796 pages)
[uploader] uploaded gluck_results_20260424_073432_2035p.zip (2578 KB, 2035 pages)
[uploader] uploaded gluck_results_20260424_074132_2271p.zip (2880 KB, 2271 pages)
[uploader] uploaded gluck_results_20260424_074834_2502p.zip (3172 KB, 2502 pages)
[uploader] uploaded gluck_results_20260424_075535_2681p.zip (3398 KB, 2681 pages)


In [3]:
upload_now()

[upload_now] OK: gluck_results_20260424_062555_1870p.zip (1870 pages, 708 KB)
